# Qwen2.5 × BBQ multi-seed sweep (letter+permute, GPU)

Runs the four **Qwen2.5** sizes (0.5B / 1.5B / 3B / 7B) in **letter + permutation**
mode for **seeds 0, 1, 2** on CUDA, so the scaling results get error bars across
seeds. Writes per-item CSVs named with the seed (e.g. `qwen7b_letterperm_cuda_seed1.csv`)
to `/kaggle/working/results/`.

**Before running:** *Settings* → **Accelerator = GPU** (16 GB T4/P100 is enough),
**Internet = On**. Optional `HF_TOKEN` Kaggle secret (Qwen2.5 is ungated).

This is 4 sizes × 3 seeds = **12 runs**; expect it to use a good chunk of the
Kaggle GPU session budget.

In [ ]:
# 1. Reduce CUDA fragmentation (helps the 7B run fit) — must be set before torch
# initializes CUDA, and it is inherited by the eval subprocesses spawned below.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clone the pipeline into an ephemeral dir (keeps /kaggle/working output = CSVs only).
!rm -rf /tmp/bias-scaling
!git clone --depth 1 https://github.com/manitawtani74/bias-scaling.git /tmp/bias-scaling

In [ ]:
# 2. Deps. Kaggle already ships CUDA torch — do NOT reinstall it.
!pip install -q -U transformers datasets accelerate

In [ ]:
# 3. Optional HF token from a Kaggle Secret named HF_TOKEN.
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    print("HF token loaded from Kaggle secret.")
except Exception as e:
    print("No HF token (fine — Qwen2.5 is ungated):", e)

In [ ]:
# 4. Confirm the GPU is visible.
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 5. Multi-seed sweep: 4 sizes × 3 seeds, letter + permutation, all on CUDA.
# Each eval runs as its own subprocess: a config that OOMs or crashes is caught
# and the sweep continues, and the subprocess exit fully releases its GPU memory.
import sys, subprocess, shutil, gc

REPO = "/tmp/bias-scaling"
WORK = "/tmp/results"            # each eval writes here first
OUT = "/kaggle/working/results"  # persisted Kaggle output
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# dtype per size: 0.5B/1.5B in float32; 3B and 7B in bfloat16 so both fit a
# single 16 GB T4 (float32 would need ~12 GB for 3B and ~28 GB for 7B).
SIZES = [("0.5B", "float32"), ("1.5B", "float32"), ("3B", "bfloat16"), ("7B", "bfloat16")]
TAG = {"0.5B": "05b", "1.5B": "15b", "3B": "3b", "7B": "7b"}
SEEDS = [0, 1, 2]

for seed in SEEDS:
    for size, dtype in SIZES:
        model = f"Qwen/Qwen2.5-{size}"
        base = f"qwen{TAG[size]}_letterperm_cuda_seed{seed}"
        work_csv = f"{WORK}/{base}.csv"
        cmd = [sys.executable, "-m", "src.evaluate",
               "--model", model, "--device", "cuda", "--dtype", dtype,
               "--scoring", "letter", "--permute",
               "--sample", "200", "--seed", str(seed), "--output", work_csv]
        print("\n>>>", " ".join(cmd), flush=True)
        try:
            subprocess.run(cmd, cwd=REPO, check=True)
            # Copy this run's two CSVs to the persisted output dir immediately, so
            # a later OOM/crash still preserves every completed run.
            for src in (work_csv, work_csv.replace(".csv", "_metrics.csv")):
                if os.path.exists(src):
                    shutil.copy(src, OUT)
                    print("saved ->", os.path.join(OUT, os.path.basename(src)), flush=True)
        except subprocess.CalledProcessError as e:
            print(f"!! {model} seed {seed} FAILED (exit {e.returncode}); continuing.", flush=True)
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [ ]:
# 6. List the CSV outputs (everything under /kaggle/working is saved on commit).
import glob
print("CSV outputs in /kaggle/working/results:")
for f in sorted(glob.glob("/kaggle/working/results/*_seed*.csv")):
    print(f"  {os.path.getsize(f):>10,d}  {os.path.basename(f)}")